# Used Car Price Valuator
## Machine Learning Regression Project

This notebook builds a **Used Car Price Valuation Model** using **Linear Regression**.

### Main Steps
1. Import libraries
2. Load dataset
3. Understand the dataset
4. Check missing values and duplicates
5. Remove duplicate records
6. Perform correlation analysis
7. Convert categorical features into numerical form
8. Separate features and target
9. Split the dataset
10. Standardize features
11. Train Linear Regression
12. Make predictions
13. Evaluate the model
14. Compare actual and predicted prices


## 1. Import Required Libraries

The following libraries are used for data handling, preprocessing, Linear Regression, and regression evaluation.


In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


## 2. Load the Dataset

Place `car_data_augmented.csv` in the same folder as this notebook.


In [45]:
df = pd.read_csv("car_data_augmented.csv")

print("First 5 Rows:")
df.head()


First 5 Rows:


,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


## 3. Dataset Shape

This shows the number of rows and columns in the dataset.


In [46]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])


Number of Rows: 451
Number of Columns: 9


## 4. Dataset Information

`info()` displays column names, data types, and non-null values.


In [47]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       451 non-null    object 
 1   Year           451 non-null    int64  
 2   Selling_Price  451 non-null    float64
 3   Present_Price  451 non-null    float64
 4   Kms_Driven     451 non-null    int64  
 5   Fuel_Type      451 non-null    object 
 6   Seller_Type    451 non-null    object 
 7   Transmission   451 non-null    object 
 8   Owner          451 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 31.8+ KB


## 5. Statistical Summary

`describe()` gives basic statistical information about the numerical columns.


In [48]:
df.describe()


,Year,Selling_Price,Present_Price,Kms_Driven,Owner
count,451.000000,451.000000,451.000000,451.000000,451.000000
mean,2012.516630,4.046718,7.941641,43885.822616,0.044346
std,3.783774,4.638668,8.382598,38341.086312,0.262945
min,2003.000000,0.100000,0.320000,500.000000,0.000000
25%,2010.000000,0.750000,1.470000,17750.000000,0.000000
50%,2014.000000,2.900000,6.790000,37000.000000,0.000000
75%,2015.000000,5.500000,10.090000,60038.000000,0.000000
max,2018.000000,35.000000,92.600000,500000.000000,3.000000


## 6. Check Missing Values

Missing values should be identified before model training.


In [49]:
print("Missing Values:")
print(df.isnull().sum())


Missing Values:
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64


## 7. Check Duplicate Records

Duplicate records can unnecessarily repeat information in the training data.


In [50]:
print("Number of Duplicate Rows:", df.duplicated().sum())


Number of Duplicate Rows: 2


## 8. Remove Duplicate Records


In [51]:
df = df.drop_duplicates()

print("Dataset Shape After Removing Duplicates:")
print(df.shape)


Dataset Shape After Removing Duplicates:
(449, 9)


In [52]:
Q1 = df["Selling_Price"].quantile(0.25)
Q3 = df["Selling_Price"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[
    (df["Selling_Price"] >= lower_bound) &
    (df["Selling_Price"] <= upper_bound)
]

## 9. Correlation Matrix

Correlation helps us understand the relationship between numerical variables.  
`Selling_Price` is the target variable for this project.


In [53]:
numeric_data = df.select_dtypes(include=np.number)

correlation_matrix = numeric_data.corr()

print("Correlation Matrix:")
correlation_matrix


Correlation Matrix:


,Year,Selling_Price,Present_Price,Kms_Driven,Owner
Year,1.000000,0.379655,-0.192340,-0.691398,-0.100186
Selling_Price,0.379655,1.000000,0.655707,-0.172978,-0.098620
Present_Price,-0.192340,0.655707,1.000000,0.212176,-0.015261
Kms_Driven,-0.691398,-0.172978,0.212176,1.000000,0.069618
Owner,-0.100186,-0.098620,-0.015261,0.069618,1.000000


## 10. Correlation with Selling Price

This displays numerical features ordered by their correlation with the target variable.


In [54]:
print("Correlation with Selling Price:")

correlation_matrix["Selling_Price"].sort_values(ascending=False)


Correlation with Selling Price:


Selling_Price    1.000000
Present_Price    0.655707
Year             0.379655
Owner           -0.098620
Kms_Driven      -0.172978
Name: Selling_Price, dtype: float64

## 11. Convert Categorical Features into Numerical Form

Machine learning models require numerical input. `pd.get_dummies()` converts categorical columns such as car name, fuel type, seller type, and transmission into numerical columns.


In [55]:
df = pd.get_dummies(
    df,
    columns=[
        "Car_Name",
        "Fuel_Type",
        "Seller_Type",
        "Transmission"
    ],
    drop_first=True,
    dtype=int
)

print("Dataset After Encoding:")
df.head()


Dataset After Encoding:


,Year,Selling_Price,Present_Price,Kms_Driven,Owner,Car_Name_Activa 3g,Car_Name_Activa 4g,Car_Name_Bajaj ct 100,Car_Name_Bajaj Avenger 150,Car_Name_Bajaj Avenger 150 street,...,Car_Name_swift,Car_Name_sx4,Car_Name_verna,Car_Name_vitara brezza,Car_Name_wagon r,Car_Name_xcent,Fuel_Type_Diesel,Fuel_Type_Petrol,Seller_Type_Individual,Transmission_Manual
0,2014,3.35,5.59,27000,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,2013,4.75,9.54,43000,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,1
2,2017,7.25,9.85,6900,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
3,2011,2.85,4.15,5200,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,1
4,2014,4.60,6.87,42450,0,0,0,0,0,0,...,1,0,0,0,0,0,1,0,0,1


## 12. Separate Features and Target

- **X** contains the input features.
- **y** contains the target variable `Selling_Price`.


In [56]:
X = df.drop("Selling_Price", axis=1)
y = df["Selling_Price"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)


Feature Shape: (429, 104)
Target Shape: (429,)


## 13. Split the Dataset into Training and Testing Sets

80% of the data is used for training and 20% for testing.


In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


Training Data Shape: (343, 104)
Testing Data Shape: (86, 104)


## 14. Feature Scaling

`StandardScaler` standardizes the input features. The scaler is fitted only on the training data and then applied to the testing data.


In [58]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature Scaling Completed")


Feature Scaling Completed


## 15. Create the Linear Regression Model


In [59]:
model = LinearRegression()

print("Linear Regression Model Created")


Linear Regression Model Created


## 16. Train the Model

The Linear Regression model learns the relationship between the input features and used-car selling price.


In [60]:
model.fit(X_train_scaled, y_train)

print("Model Training Completed")


Model Training Completed


## 17. Make Predictions

The trained model predicts selling prices for the test data.


In [61]:
y_pred = model.predict(X_test_scaled)

print("First 10 Predictions:")
print(y_pred[:10])


First 10 Predictions:
[6.84108686 5.00571983 2.49000583 1.64977147 2.31968861 4.51474726
 3.09535614 0.75420359 1.15457193 1.22148528]


## 18. Evaluate Model Performance

The model is evaluated using:

- **MAE** – Mean Absolute Error
- **MSE** – Mean Squared Error
- **RMSE** – Root Mean Squared Error
- **R² Score** – indicates how well the model explains variation in selling price


In [62]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("==============================")
print("MODEL PERFORMANCE")
print("==============================")
print("MAE      :", mae)
print("MSE      :", mse)
print("RMSE     :", rmse)
print("R2 Score :", r2)


MODEL PERFORMANCE
MAE      : 0.9527340668033284
MSE      : 1.4007679522981342
RMSE     : 1.183540431205514
R2 Score : 0.8036349633845996


## 19. Actual Price vs Predicted Price

The following table compares the actual selling prices with the values predicted by the Linear Regression model.


In [63]:
result = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

result.head(10)


,Actual Price,Predicted Price
0,7.91,6.841087
1,3.80,5.005720
2,0.18,2.490006
3,2.35,1.649771
4,1.84,2.319689
5,5.40,4.514747
6,0.40,3.095356
7,0.40,0.754204
8,0.27,1.154572
9,0.20,1.221485


## Conclusion

The Used Car Price Valuator uses **Linear Regression** to estimate the selling price of a used car based on the available vehicle features. The model performance is measured using MAE, MSE, RMSE, and R² Score.
